# Retrain Refinement for One Area

Reuse saved proposal rasters to train a replacement refinement model with revised crown annotations.


## Inputs

Complete a persistent one-area training run first, as shown in [Train and Segment One Area](../train_and_segment_one_area/).
This notebook uses the training proposals saved under the source model's _training/training_run folder, the same source rasters, and a revised crown GeoPackage.
The paths below refer to your own data and training outputs.
The resulting refinement model is written to a separate directory.


In [ ]:
import subprocess
import sys
from pathlib import Path

from samstars import SamStars
from samstars.data import build_scale_stats_from_proposal_records, build_single_tile
from samstars.training.records import build_stardist_training_records
from samstars.training.stardist_training import train_stardist_model


## Set Paths

Use the same training rasters that produced the saved proposals, with the revised annotations covering that area.


In [ ]:
source_model = SamStars.load(Path("models/crowns").resolve())
training_dir = Path("data/training").resolve()
crowns = training_dir / "crowns_revised.gpkg"
proposal_records = source_model.root / "_training/training_run/proposals/proposal_records.json"

work_dir = Path("runs/refinement_update").resolve()
labeled_tiles = work_dir / "labeled_tiles"
records_dir = work_dir / "training_records"
refinement_model = work_dir / "models/refinement"
scale_stats = work_dir / "training_scale_stats.json"
outputs_dir = work_dir / "crowns"


## Prepare the Revised Labels

The prepared tile uses the original model's band settings. Its directory name must match the source training tile so records can be associated with the revised annotations.


In [ ]:
build_single_tile(
    img=training_dir / "image.tif",
    chm=training_dir / "chm.tif",
    den=training_dir / "first_return_density.tif",
    crowns=crowns,
    out=labeled_tiles,
    rgb_bands=source_model.defaults.get("rgb_bands"),
    nir_band=source_model.defaults.get("nir_band"),
    texture_band=source_model.defaults.get("texture_band"),
    skip_existing=False,
)


## Build Refinement Training Records

These records pair existing proposal rasters with masks from the revised crown polygons. Use a new work directory for each annotation revision.


In [ ]:
train_records, val_records = build_stardist_training_records(
    proposal_records=proposal_records,
    tiles_dir=labeled_tiles,
    out_dir=records_dir,
    val_fraction=0.2,
)

build_scale_stats_from_proposal_records(
    proposal_records=proposal_records,
    out=scale_stats,
)


## Train the Replacement Model


In [ ]:
train_stardist_model(
    train_records=train_records,
    val_records=val_records,
    out_dir=refinement_model.parent,
    model_name=refinement_model.name,
    epochs=30,
    input_key="index",
    use_aux=True,
    scale_stats_path=scale_stats,
)


## Run Refinement

This command applies the replacement model to the saved proposals and writes crown polygons. It uses the model's learned thresholds when available.


In [ ]:
subprocess.run([
    sys.executable, "-m", "samstars.segmentation.refinement",
    "--proposal-records", str(proposal_records),
    "--model-dir", str(refinement_model.parent),
    "--model-name", refinement_model.name,
    "--out-dir", str(outputs_dir),
    "--input-key", "index",
    "--use-aux",
    "--scale-stats", str(scale_stats),
    "--write-labels",
], check=True)

print(outputs_dir / "crowns.gpkg")


## Further Runs

The replacement is a standalone refinement model. To use it alongside the original decoder on prepared inference tiles, see [Segment With Separate Model Files](../advanced_workflows/#segment-with-separate-model-files).
